# Кандидатогенерация объявлений для поисковых запросов Авито



## 1. Импорт библиотек и параметры


In [1]:
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from transformers import AutoModel, AutoTokenizer

DATA_DIR = Path('.')
TOP_K = 50
RANDOM_STATE = 42
started_at = time.time()
np.random.seed(RANDOM_STATE)

## 2. Загрузка данных


In [2]:
train_columns = [
    'search_query', 'search_location_id', 'search_infm_params_text',
    'search_category', 'item_id', 'item_microcat_id'
]
train = pd.read_parquet(DATA_DIR / 'train.parquet', columns=train_columns)
queries = pd.read_parquet(DATA_DIR / 'benchmark_queries.parquet')
items = pd.read_parquet(DATA_DIR / 'benchmark_items.parquet')
train['item_id'] = train['item_id'].astype(str)
queries['query_id'] = queries['query_id'].astype(str)
items['item_id'] = items['item_id'].astype(str)
print(f'train: {train.shape}')
print(f'queries: {queries.shape}')
print(f'items: {items.shape}')

train: (497673, 6)
queries: (2452, 6)
items: (189212, 14)


## 3. Нормализация текста

In [3]:
def normalize_text(series):
    return (
        series.fillna('').astype(str).str.lower()
        .str.replace('ё', 'е', regex=False)
        .str.replace(r'[^0-9a-zа-я]+', ' ', regex=True)
        .str.strip()
    )

item_title = normalize_text(items['item_title_raw'])
item_params = normalize_text(items['item_infm_params_text']).str.slice(0, 650)
item_description = normalize_text(items['item_description_raw']).str.slice(0, 650)
query_text = normalize_text(queries['search_query'])
query_params = normalize_text(queries['search_infm_params_text'])
train['query_norm'] = normalize_text(train['search_query'])
queries['query_norm'] = query_text
item_documents = item_title + ' ' + item_title + ' ' + item_params + ' ' + item_description
query_documents = query_text + ' ' + query_text + ' ' + query_text + ' ' + query_params

## 4. Статистики обучающих переходов


In [4]:
item_ids = items['item_id'].to_numpy()
item_id_to_pos = {item_id: pos for pos, item_id in enumerate(item_ids)}
corpus_ids = set(item_ids)
train_in_corpus = train[train['item_id'].isin(corpus_ids)].copy()

query_item_counts = (
    train_in_corpus.groupby(['query_norm', 'item_id'], sort=False)
    .size().rename('count').reset_index()
    .sort_values(['query_norm', 'count'], ascending=[True, False])
)
query_item_map = {
    key: list(zip(group['item_id'].head(50), group['count'].head(50)))
    for key, group in query_item_counts.groupby('query_norm', sort=False)
}

location_item_counts = (
    train_in_corpus.groupby(['query_norm', 'search_location_id', 'item_id'], sort=False)
    .size().rename('count').reset_index()
    .sort_values(['query_norm', 'search_location_id', 'count'], ascending=[True, True, False])
)
location_item_map = {
    key: list(zip(group['item_id'].head(50), group['count'].head(50)))
    for key, group in location_item_counts.groupby(['query_norm', 'search_location_id'], sort=False)
}

microcat_counts = (
    train.groupby(['query_norm', 'item_microcat_id'], sort=False)
    .size().rename('count').reset_index()
    .sort_values(['query_norm', 'count'], ascending=[True, False])
)
microcat_map = {
    key: group['item_microcat_id'].head(3).to_numpy()
    for key, group in microcat_counts.groupby('query_norm', sort=False)
}

popularity_counts = train_in_corpus['item_id'].value_counts()
item_popularity = items['item_id'].map(popularity_counts).fillna(0).to_numpy(dtype=np.float32)
item_popularity = np.log1p(item_popularity)
if item_popularity.max() > 0:
    item_popularity /= item_popularity.max()
reviews = np.log1p(items['item_rating_reviews_count'].fillna(0).clip(lower=0).to_numpy(dtype=np.float32))
if reviews.max() > 0:
    reviews /= reviews.max()
ratings = items['item_rating'].fillna(0).clip(lower=0, upper=5).to_numpy(dtype=np.float32) / 5
fallback_quality = 0.7 * item_popularity + 0.2 * reviews + 0.1 * ratings

## 5. Построение текстовых индексов


In [5]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_df=0.7, max_features=160000,
    sublinear_tf=True, dtype=np.float32
)
item_word_matrix = word_vectorizer.fit_transform(item_documents)
query_word_matrix = word_vectorizer.transform(query_documents)
print(f'word matrix: {item_word_matrix.shape}, nnz={item_word_matrix.nnz:,}')

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=140000,
    sublinear_tf=True, dtype=np.float32
)
item_char_matrix = char_vectorizer.fit_transform(item_title)
query_char_matrix = char_vectorizer.transform(query_text)
print(f'char matrix: {item_char_matrix.shape}, nnz={item_char_matrix.nnz:,}')

word matrix: (189212, 160000), nnz=30,512,049
char matrix: (189212, 97143), nnz=13,575,707


## 6. Нейросетические семантические эмбеддинги


In [6]:
torch.set_num_threads(4)
model_path = DATA_DIR / 'rubert-tiny2'
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
semantic_model = AutoModel.from_pretrained(model_path, local_files_only=True).eval()

def encode_texts(texts, batch_size=128, max_length=48):
    embeddings = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            hidden = semantic_model(**batch).last_hidden_state
            mask = batch['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1)
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            embeddings.append(pooled.cpu().numpy().astype(np.float32))
    return np.vstack(embeddings)

item_semantic_text = (items['item_title_raw'].fillna('') + ' [SEP] ' + items['item_infm_params_text'].fillna('').str.slice(0, 220)).tolist()
query_semantic_text = (queries['search_query'].fillna('') + ' [SEP] ' + queries['search_infm_params_text'].fillna('')).tolist()
embedding_cache = DATA_DIR / 'rubert_item_embeddings.npy'
if embedding_cache.exists():
    item_embeddings = np.load(embedding_cache)
else:
    item_embeddings = encode_texts(item_semantic_text)
    np.save(embedding_cache, item_embeddings)
assert item_embeddings.shape == (len(items), 312)
query_embeddings = encode_texts(query_semantic_text)
print(f'semantic embeddings: items={item_embeddings.shape}, queries={query_embeddings.shape}')


Loading weights: 100%|##########| 55/55 [00:00<00:00, 2571.27it/s, Materializing param=pooler.dense.weight]
semantic embeddings: items=(189212, 312), queries=(2452, 312)


## 7. Извлечение кандидатов


In [7]:
item_locations = items['item_location_id'].to_numpy()
item_categories = items['item_category_id'].to_numpy()
item_microcats = items['item_microcat_id'].to_numpy()
query_locations = queries['search_location_id'].to_numpy()
query_categories = queries['search_category'].to_numpy()

def best_indices(indices, values, mask, limit):
    selected = np.flatnonzero(mask)
    if selected.size == 0:
        return np.empty(0, dtype=np.int32)
    if selected.size > limit:
        local = np.argpartition(values[selected], -limit)[-limit:]
        selected = selected[local]
    selected = selected[np.argsort(values[selected])[::-1]]
    return indices[selected].astype(np.int32, copy=False)

def retrieve_candidates(query_matrix, item_matrix, global_limit=250, local_limit=250, micro_limit=120, batch_size=80):
    result = []
    for start in range(0, query_matrix.shape[0], batch_size):
        similarities = (query_matrix[start:start + batch_size] @ item_matrix.T).tocsr()
        for offset in range(similarities.shape[0]):
            query_pos = start + offset
            left = similarities.indptr[offset]
            right = similarities.indptr[offset + 1]
            indices = similarities.indices[left:right]
            values = similarities.data[left:right]
            category = query_categories[query_pos]
            category_mask = np.ones(indices.size, dtype=bool) if category == 0 else item_categories[indices] == category
            global_part = best_indices(indices, values, category_mask, global_limit)
            local_mask = category_mask & (item_locations[indices] == query_locations[query_pos])
            local_part = best_indices(indices, values, local_mask, local_limit)
            predicted_microcats = microcat_map.get(queries.iloc[query_pos]['query_norm'], np.empty(0, dtype=np.int64))
            micro_mask = category_mask & np.isin(item_microcats[indices], predicted_microcats)
            micro_part = best_indices(indices, values, micro_mask, micro_limit)
            result.append(np.unique(np.concatenate([global_part, local_part, micro_part])))
    return result

word_candidates = retrieve_candidates(query_word_matrix, item_word_matrix)
char_candidates = retrieve_candidates(query_char_matrix, item_char_matrix)

def dense_best(values, mask, limit):
    selected = np.flatnonzero(mask)
    if selected.size > limit:
        selected = selected[np.argpartition(values[selected], -limit)[-limit:]]
    return selected.astype(np.int32, copy=False)

semantic_candidates = []
for start in range(0, len(queries), 32):
    similarities = query_embeddings[start:start + 32] @ item_embeddings.T
    for offset, values in enumerate(similarities):
        query_pos = start + offset
        category = query_categories[query_pos]
        category_mask = np.ones(len(items), dtype=bool) if category == 0 else item_categories == category
        global_part = dense_best(values, category_mask, 250)
        local_part = dense_best(values, category_mask & (item_locations == query_locations[query_pos]), 250)
        predicted_microcats = microcat_map.get(queries.iloc[query_pos]['query_norm'], np.empty(0, dtype=np.int64))
        micro_part = dense_best(values, category_mask & np.isin(item_microcats, predicted_microcats), 120)
        semantic_candidates.append(np.unique(np.concatenate([global_part, local_part, micro_part])))

## 8. Объединение и ранжирование


In [8]:
predictions = []
all_positions = np.arange(len(items), dtype=np.int32)

for query_pos, row in queries.iterrows():
    query_norm = row['query_norm']
    location = row['search_location_id']
    category = row['search_category']
    direct = query_item_map.get(query_norm, [])
    local_direct = location_item_map.get((query_norm, location), [])
    direct_positions = [item_id_to_pos[item_id] for item_id, count in direct]
    local_direct_positions = [item_id_to_pos[item_id] for item_id, count in local_direct]
    candidates = np.unique(np.concatenate([
        word_candidates[query_pos], char_candidates[query_pos], semantic_candidates[query_pos],
        np.asarray(direct_positions, dtype=np.int32),
        np.asarray(local_direct_positions, dtype=np.int32)
    ]))
    if category != 0:
        candidates = candidates[item_categories[candidates] == category]
    if candidates.size < TOP_K:
        fallback_mask = item_locations == location
        if category != 0:
            fallback_mask &= item_categories == category
        fallback_pool = all_positions[fallback_mask]
        fallback_order = fallback_pool[np.argsort(fallback_quality[fallback_pool])[::-1]]
        candidates = np.unique(np.concatenate([candidates, fallback_order[:TOP_K * 3]]))
    if candidates.size < TOP_K:
        fallback_mask = np.ones(len(items), dtype=bool)
        if category != 0:
            fallback_mask &= item_categories == category
        fallback_pool = all_positions[fallback_mask]
        fallback_order = fallback_pool[np.argsort(fallback_quality[fallback_pool])[::-1]]
        candidates = np.unique(np.concatenate([candidates, fallback_order[:TOP_K * 3]]))
    word_score = (query_word_matrix[query_pos] @ item_word_matrix[candidates].T).toarray().ravel()
    char_score = (query_char_matrix[query_pos] @ item_char_matrix[candidates].T).toarray().ravel()
    semantic_score = query_embeddings[query_pos] @ item_embeddings[candidates].T
    score = 0.35 * word_score + 0.10 * char_score + 0.55 * semantic_score
    score += 0.20 * (item_locations[candidates] == location)
    predicted_microcats = microcat_map.get(query_norm, np.empty(0, dtype=np.int64))
    score += 0.10 * np.isin(item_microcats[candidates], predicted_microcats)
    score += 0.02 * item_popularity[candidates]
    candidate_lookup = {position: offset for offset, position in enumerate(candidates)}
    for item_id, count in direct:
        position = item_id_to_pos[item_id]
        if position in candidate_lookup:
            score[candidate_lookup[position]] += 0.30 + 0.02 * np.log1p(count)
    for item_id, count in local_direct:
        position = item_id_to_pos[item_id]
        if position in candidate_lookup:
            score[candidate_lookup[position]] += 0.30 + 0.02 * np.log1p(count)
    order = np.argsort(score)[::-1][:TOP_K]
    predictions.append(item_ids[candidates[order]].tolist())

lengths = pd.Series([len(x) for x in predictions])
print(lengths.describe().to_string())

count    2452.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0


## 9. Формирование и проверка ответа

In [ ]:
answer = pd.DataFrame({
    'query_id': queries['query_id'],
    'answer': [' '.join(prediction) for prediction in predictions]
})

answer.to_csv(DATA_DIR / 'answer.csv', index=False, encoding='utf-8')


        query_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            answer
70DfDUpwjxB4lzFd d01e14d2ffd4ddba d722bcda1a555091 603623b4bd9e8f6c 4494aa11cbdc2d65 e08f4ce94c268cef 936c0eb79b64074d d7ad9f91504e4d